In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
from functools import reduce
import pylab as plt
from statannot import add_stat_annotation

from importlib import reload
import itertools
from itertools import count

import sys

sys.path.insert(1,'../../make_figures')
import plots

In [ ]:
data_path = '/scratch/c.c21013066/data/ukbiobank'

# Baseline info: verbal interview on drugs

In [ ]:
# load drugs
drugs = pd.read_csv(f'{data_path}/phenotypes/AntiParkinsonism.csv')
drugs2 = pd.read_csv(f'{data_path}/phenotypes/Dopa.csv')

drugs = pd.merge(drugs,drugs2,on='eid')

In [ ]:
cases = pd.read_csv(f'{data_path}/sample/withGP/noOsteo/ParkinsonDisease.csv')
cases['ParkinsonDisease'] = 1
cases['time_to_diagnosis'] = cases['ParkinsonDisease_age'] - cases['visit_age']

cases2 = pd.read_csv(f'{data_path}/sample/withGP/noOsteo/AllCauseParkinsonism.csv')
cases2['AllCauseParkinsonism'] = 1
cases2['time_to_diagnosis_AllCauseParkinsonism'] = cases2['AllCauseParkinsonism_age'] - cases2['visit_age']

In [ ]:
m = pd.merge(cases,drugs,on='eid',how='outer')
m.loc[m['ParkinsonDisease']!=1,'ParkinsonDisease'] = 0
m = pd.merge(cases2,m,on='eid',how='outer')
m.loc[m['AllCauseParkinsonism']!=1,'AllCauseParkinsonism'] = 0

In [ ]:
3837/100*15.95

In [ ]:
m.groupby(['AllCauseParkinsonism','ParkinsonDisease'])[['antiparkinsonism','Dopa']].agg(['size','mean','sum'])

In [ ]:
m.groupby('antiparkinsonism')['visit_age_x','ParkinsonDisease_age','time_to_diagnosis'].agg(['mean','size','std'])

# GP prescription

In [ ]:
gp = pd.read_csv(f'{data_path}/record_level/gp_AntiParkinsonism_raw.csv',parse_dates=['issue_date'])
gp['treated'] = 'True'
gp_first = pd.read_csv(f'{data_path}/record_level/gp_AntiParkinsonism_first.csv',parse_dates=['gp_AntiParkinsonism_date'])

In [ ]:
gp

In [ ]:
m = pd.merge(cases,gp_first,on='eid',how='outer')
m.loc[m['ParkinsonDisease']!=1,'ParkinsonDisease'] = 0
#m = pd.merge(cases2,m,on='eid',how='outer',suffixes=['_allcausepd',''])
#m.loc[m['AllCauseParkinsonism']!=1,'AllCauseParkinsonism'] = 0
#m.loc[m['gp_AntiParkinsonism'].isna(),'gp_AntiParkinsonism'] = 0

In [ ]:
(3837-2045)/100*11.4955, 100/3837*206, 3837-206-1586

In [ ]:
m.groupby(['ParkinsonDisease'])[['gp_AntiParkinsonism']].agg(['size',np.mean,'sum',lambda x: x.isna().sum()])

In [ ]:
(100/cases.shape[0])*206

In [ ]:
m[m['ParkinsonDisease']==1].groupby('gp_AntiParkinsonism')['visit_age','ParkinsonDisease_age','time_to_diagnosis'].agg(['mean','size','std','min','max'])

In [ ]:
m.groupby(['antiparkinsonism','gp_AntiParkinsonism']).size()

In [ ]:
fig = plt.figure(figsize=(10,5))
sns.lineplot(x='issue_date',y='quantity',data=gp[gp['eid']==1030610])

# Analysis whether medication has effect on acceleration

In [ ]:
# used during accelerometry?
data_raw = pd.read_csv(f'{data_path}/phenotypes/accelerometer/unmatched_all_HCnoOsteo.csv',parse_dates=['date_accelerometry'])
res = pd.read_csv(f'{data_path}/phenotypes/accelerometer/unmatched_all_residual_BMI_HCnoOsteo_intercept.csv')
acc = pd.read_csv(f'{data_path}/phenotypes/demo_acc.csv',parse_dates=['date_accelerometry','date_visit'])
data = pd.merge(data_raw,acc[['eid','date_visit','date_accelerometry']],on='eid',how='outer',suffixes=['_drop',''])
gp = pd.read_csv(f'{data_path}/record_level/gp_AntiParkinsonism_raw.csv',parse_dates=['issue_date'],dayfirst=True)
gp['treated'] = 'True'
gp_first = pd.read_csv(f'{data_path}/record_level/gp_AntiParkinsonism_first.csv',parse_dates=['gp_AntiParkinsonism_date'])
data = pd.merge(data,res[['eid','average acceleration residual_bmi']],on='eid',how='left')
data = pd.merge(data,gp_first[['eid','gp_AntiParkinsonism']],on='eid',how='left')
# cases = pd.read_csv(f'{data_path}/sample/withGP/noOsteo/ParkinsonDisease.csv')
# cases['ParkinsonDisease'] = 1
# cases['time_to_diagnosis'] = cases['ParkinsonDisease_age'] - cases['visit_age']
# data = pd.merge(data,cases[['eid','ParkinsonDisease']],on='eid',how='left')
data = data.dropna(subset=['date_accelerometry'])

In [ ]:
# get closest date to acc_date
data = data.sort_values('date_accelerometry')
gp = gp.sort_values('issue_date')
accm = pd.merge_asof(data,gp,by='eid',left_on='date_accelerometry',right_on='issue_date',tolerance=pd.Timedelta(weeks=10),direction='backward')

accm.loc[np.logical_and(accm['treated'].isna(),~accm['gp_AntiParkinsonism'].isna()),'treated'] = 'False'
accm.loc[np.logical_and(accm['treated'].isna(),accm['gp_AntiParkinsonism'].isna()),'treated'] = 'None'

In [ ]:
accm['acc_time_since_medication'] = accm['date_accelerometry'] - accm['issue_date']
accm['acc_time_to_medication'] = accm['issue_date'] - accm['date_accelerometry']

accm['acc_time_since_medication_days'] = (accm['date_accelerometry'] - accm['issue_date']) / pd.Timedelta(days=1)
accm['acc_time_to_medication_days'] = (accm['issue_date'] - accm['date_accelerometry']) / pd.Timedelta(days=1)

In [ ]:
accm_back = accm.groupby(['diagnosis','eid']).first() # for when direction == backward
#accm_first = accm.groupby(['diagnosis','eid']).first() # for when direction == nearest

In [ ]:
accm_back.filter(regex='acc_time').describe()

In [ ]:
difference = np.setdiff1d(accm_first[~accm_first['acc_time_since_medication'].isna()].index,accm_back[~accm_back['acc_time_since_medication'].isna()].index)

In [ ]:
accm_first.loc[difference].filter(regex='date'),accm_first.loc[difference].groupby('diagnosis')[['acc_time_to_medication_days']].agg(['size','count','mean','min','max','std'])

In [ ]:
accm
accm_back.groupby(['diagnosis','Status'])[['acc_time_since_medication_days','gp_AntiParkinsonism']].agg(['size','count','mean','std',lambda x: x.isna().sum()])

In [ ]:
ax = sns.boxplot(hue='Status',x='treated',y='No_wear_time_bias_adjusted_average_acceleration',data=accm_back.loc['ParkinsonDisease'],showfliers=False)
ax.set_ylabel('average acceleration')

In [ ]:
fig,axes = plt.subplots(ncols=2,figsize=(12,5))
plots.plot_context()
ax0 = sns.boxplot(x='treated',y='No_wear_time_bias_adjusted_average_acceleration',data=accm_back[accm_back['Status']=='Diseased'].loc['ParkinsonDisease'],showfliers=False,color='k',ax=axes[0])
test_results = add_stat_annotation(ax0, data=accm_back[accm_back['Status']=='Diseased'].loc['ParkinsonDisease'], x='treated', y='No_wear_time_bias_adjusted_average_acceleration',
                                   test='Mann-Whitney', text_format='star',box_pairs = [("True",'False')],
                                   loc='outside', verbose=2)
ax0.set_ylabel('average acceleration')
ax0.text(-0.22, 1.01,'A', transform=ax0.transAxes,size=20, weight='bold')
N_diagnosis = accm_back[accm_back['Status']=='Diseased'].loc['ParkinsonDisease'].dropna(subset=['No_wear_time_bias_adjusted_average_acceleration'])['treated'].value_counts()
plots.add_median_labels(ax0,N_diagnosis,fmt="%d",remove=1)

ax1 = sns.boxplot(x='treated',y='average acceleration residual_bmi',data=accm_back[accm_back['Status']=='Diseased'].loc['ParkinsonDisease'],showfliers=False,color='k',ax=axes[1])
test_results = add_stat_annotation(ax1, data=accm_back[accm_back['Status']=='Diseased'].loc['ParkinsonDisease'], x='treated', y='No_wear_time_bias_adjusted_average_acceleration',
                                   test='Mann-Whitney', text_format='star',box_pairs = [("True",'False')],
                                   loc='outside', verbose=2)
ax1.set_ylabel('average acceleration residual')
ax1.text(-0.22, 1.01,'B', transform=ax1.transAxes,size=20, weight='bold')
N_diagnosis = accm_back[accm_back['Status']=='Diseased'].loc['ParkinsonDisease'].dropna(subset=['average acceleration residual_bmi'])['treated'].value_counts()
plots.add_median_labels(ax1,N_diagnosis,fmt="%d",remove=1)
plt.tight_layout()
plt.savefig('/scratch/c.c21013066/images/paper/ProdromalUKBB/medication_acceleration_NaNtreated.png',dpi=300,bbox_inches='tight')
plt.savefig('/scratch/c.c21013066/images/paper/ProdromalUKBB/medication_acceleration_NaNtreated.pdf',dpi=300,bbox_inches='tight')